# Reciter TTS — Qwen3-TTS 0.6B → ONNX (AR pipeline) → Android

**Это новый, валидированный авторегрессионный (AR) пайплайн.** Старый одно-проходный
экспорт талкера был архитектурно неверным (талкер — это AR-декодер по словарю
кодеков, а не по тексту). Здесь экспортируются правильные блоки и они численно
совпадают с `model.generate` **бит-в-бит** (коды 100%, аудио идентично).

Что собирается (см. `docs/TALKER_AR_PLAN.md`, `docs/PIPELINE.md`):

| файл | что |
|---|---|
| `talker_step.onnx` | 28-слойный талкер, 1 шаг, KV-кэш → logits[3072]+hidden+KV |
| `subtalker_step.onnx` | 5-слойный code predictor, 1 шаг, KV-кэш |
| `codec_embed.onnx` | code0 (0..3071) → embedding[1024] |
| `code2wav.onnx` | codes[1,16,T] → аудио 24 кГц |
| `text_cond_table.f16` | [151936,1024] = text_projection(text_embedding) |
| `subtalker_codec_embed.f16` / `subtalker_heads.f16` | [15,2048,1024] |
| `baked_voices.bin` + `voices.json` | русские голоса (x-векторы) |

Три «тяжёлых» ONNX автоматически квантуются в INT8 (≈1.7 ГБ → ≈700 МБ).


## Ячейка 1: Установка зависимостей (чисто; БЕЗ onnxscript — используется legacy-экспорт)

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.57.3 accelerate sentencepiece protobuf huggingface-hub qwen-tts
!pip install -q onnx onnxruntime librosa soundfile numpy

import os, json, numpy as np, torch
print('torch', torch.__version__)


## Ячейка 2: Загрузка модели (fp32 на CPU — нужно для точного экспорта)

In [ ]:
from qwen_tts import Qwen3TTSModel
MODEL_NAME = 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
model_wrapper = Qwen3TTSModel.from_pretrained(MODEL_NAME, device_map='cpu', dtype=torch.float32)
model = model_wrapper.model
model.eval()
print('loaded', type(model).__name__)


## Ячейка 3: Экспорт всех ONNX-блоков + INT8-квантование

Запускает `tools/export_talker_ar.py` из репозитория. Скрипт экспортирует
`text_cond_table.f16`, `codec_embed.onnx`, `talker_step.onnx`,
`subtalker_step.onnx` (+ raw fp16 таблицы), `code2wav.onnx`, валидирует каждый
блок численно, затем квантует три тяжёлых графа в INT8.


In [ ]:
import urllib.request
os.makedirs('tools', exist_ok=True)
urllib.request.urlretrieve('%s/tools/export_talker_ar.py' % 'https://raw.githubusercontent.com/sdnpty/reciter-tts/main', 'tools/export_talker_ar.py')
exec(open('tools/export_talker_ar.py').read())  # uses `model` from cell 2; writes /content/qwen3-tts-onnx/android
OUT = '/content/qwen3-tts-onnx/android'


## Ячейка 4: Русские голоса (FLEURS, CC-BY) — без блокирующих виджетов

Качает русские клипы из **google/fleurs** автоматически. Свои клипы:
положите WAV в `/content/ref` через панель Files слева ДО запуска — тогда
FLEURS пропускается. Никаких всплывающих диалогов (поэтому не зависает).


In [ ]:
# Референсные голоса для запекания. По умолчанию берём те же 5 клипов из
# репозитория (tools/refs), что и для F5 — чтобы голоса в обеих моделях
# совпадали. Свои клипы: положи WAV в /content/ref ДО запуска (перекроют).
import os, glob, urllib.request
REF = '/content/ref'; os.makedirs(REF, exist_ok=True)

have = [f for f in os.listdir(REF) if f.lower().endswith(('.wav','.ogg','.mp3','.flac'))]
if have:
    print('Использую уже загруженные клипы:', have)
else:
    base = 'https://raw.githubusercontent.com/sdnpty/reciter-tts/main/tools/refs'
    for n in range(1, 6):
        try:
            urllib.request.urlretrieve(f'{base}/ref{n}.wav', os.path.join(REF, f'ref{n}.wav'))
            print(f'  + ref{n}.wav')
        except Exception as e:
            print(f'  ! ref{n}.wav: {e}')

clips = sorted([f for f in os.listdir(REF) if f.lower().endswith(('.wav','.ogg','.mp3','.flac'))])
assert clips, 'Нет клипов. Положи русские WAV в /content/ref и перезапусти.'
print('reference clips:', clips)


## Ячейка 5: Запекание голосов (x-векторы)

Для каждого клипа считаем x-вектор спикера тем же путём, что и `model.generate`
в режиме `x_vector_only_mode=True`. Сохраняем `baked_voices.bin`
(склеенные float32[1024]) + `voices.json` (имена).


In [ ]:
import librosa, numpy as np, torch, os, onnxruntime as ort

# Запекание speaker x-векторов.
# ВАЖНО: talker ждёт x-вектор РОВНО в том пространстве, что выдаёт нативный
# generate_speaker_prompt модели. На устройстве было слышно "гул" без EOS именно
# потому, что раньше ячейка молча откатывалась на speaker_encoder.onnx (другое
# пространство). Теперь нативный путь — основной, а энкодер — только по явному
# флагу и с предупреждением.
ALLOW_ENCODER_FALLBACK = False   # поставь True ТОЛЬКО если нативного пути нет

def _extract(out, dim=1024):
    cands = []
    def walk(o):
        if isinstance(o, torch.Tensor): cands.append(o)
        elif isinstance(o, (tuple, list)):
            for x in o: walk(x)
        elif isinstance(o, dict):
            for x in o.values(): walk(x)
    walk(out)
    for t in cands:
        if dim in tuple(t.shape):
            return t.reshape(-1)[:dim].float().cpu().numpy().astype(np.float32)
    return cands[0].reshape(-1)[:dim].float().cpu().numpy().astype(np.float32) if cands else None

_gsp = getattr(model, 'generate_speaker_prompt', None) or \
       getattr(globals().get('model_wrapper', None), 'generate_speaker_prompt', None)
print('generate_speaker_prompt:', 'НАЙДЕН' if _gsp else 'НЕ НАЙДЕН')

_se_sess = None
_se_path = f'{OUT}/speaker_encoder.onnx'
if os.path.exists(_se_path):
    _se_sess = ort.InferenceSession(_se_path, providers=['CPUExecutionProvider'])

import inspect
if _gsp is not None:
    try: print('сигнатура generate_speaker_prompt:', inspect.signature(_gsp))
    except Exception as e: print('сигнатуру не получить:', e)

def native_xvector(wt):
    """Нативный x-вектор модели (правильное пространство). Печатает реальную
    причину каждой неудачной попытки, чтобы подобрать верный вызов."""
    if _gsp is None: return None
    wav_np = wt.squeeze(0).cpu().numpy()
    attempts = [
        ('_gsp(wt)',                   lambda: _gsp(wt)),
        ('_gsp(wt, 16000)',            lambda: _gsp(wt, 16000)),
        ('_gsp(wt, sample_rate=16000)',lambda: _gsp(wt, sample_rate=16000)),
        ('_gsp(wt[0])',                lambda: _gsp(wt[0])),
        ('_gsp(wav_np)',               lambda: _gsp(wav_np)),
        ('_gsp(wav_np, 16000)',        lambda: _gsp(wav_np, 16000)),
    ]
    for label, call in attempts:
        try:
            with torch.no_grad(): out = call()
            v = _extract(out)
            if v is not None and v.shape[0] == 1024:
                print('  OK через', label)
                return v
            print('  ', label, '-> извлёк форму', None if v is None else v.shape)
        except Exception as e:
            print('  ', label, '-> ошибка:', repr(e)[:160])
    return None

def speaker_xvector(wav_path):
    wav, _ = librosa.load(wav_path, sr=16000, mono=True)
    wt = torch.from_numpy(wav).float().unsqueeze(0)
    v = native_xvector(wt)
    if v is not None: return v, 'native'
    if ALLOW_ENCODER_FALLBACK and _se_sess is not None:
        ev = _se_sess.run(None, {'waveform': wav[None].astype(np.float32)})[0].reshape(-1)[:1024].astype(np.float32)
        return ev, 'encoder(ОПАСНО)'
    raise RuntimeError(
        'Нативный generate_speaker_prompt недоступен. Энкодер-фоллбэк отключён, '
        'т.к. даёт неверное пространство (гул/нет EOS на устройстве). '
        'Найди правильный вызов: [n for n in dir(model) if "speaker" in n.lower() or "prompt" in n.lower()] '
        'и поправь native_xvector(), либо в крайнем случае ALLOW_ENCODER_FALLBACK=True.')

names, vecs, methods = [], [], set()
for c in clips:
    name = os.path.splitext(c)[0]
    v, how = speaker_xvector(os.path.join(REF, c))
    assert v.shape == (1024,), v.shape
    names.append(name); vecs.append(v); methods.add(how)
    print(f'  baked {name}: |x|={np.linalg.norm(v):.2f} std={v.std():.4f} [{how}]')

np.stack(vecs).astype(np.float32).tofile(f'{OUT}/baked_voices.bin')
json.dump({'voices': names, 'dim': 1024}, open(f'{OUT}/voices.json','w'), ensure_ascii=False, indent=2)
print('voices:', names, '| способ:', methods)
if 'native' not in methods:
    print('!!! ВНИМАНИЕ: голоса запечены НЕ нативным путём — на устройстве будет гул.')


## Ячейка 6: ar_config.json + токенизатор

Сохраняет токены роли/суффикса (захвачены из `model.generate`) и файлы
токенизатора, которые читает приложение.


In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok.save_pretrained(f'{OUT}/tokenizer_tmp')
import shutil
for f in ['vocab.json','merges.txt']:
    src = f'{OUT}/tokenizer_tmp/{f}'
    if os.path.exists(src): shutil.copy(src, f'{OUT}/{f}')

ar_config = {
    'role_tokens':   [151644, 77091, 198],
    'suffix_tokens': [151645, 198, 151644, 77091, 198],
    'voices': names,
    'sample_rate': 24000,
    'language_id': 2069,  # russian
    'arch': 'qwen3-tts-ar',
}
json.dump(ar_config, open(f'{OUT}/ar_config.json','w'), ensure_ascii=False, indent=2)
print(ar_config)

# model.json — manifest the Android app reads (ModelConfig.parseManifestJson)
model_json = {
    'id': 'qwen3-tts-ar', 'displayName': 'Qwen3 TTS 0.6B (AR)',
    'family': 'qwen3-tts', 'architecture': 'qwen3-codec-ar',
    'sampleRateHz': 24000, 'codecFrameRateHz': 12,
    'files': [
        {'filename': 'talker_step.onnx',    'role': 'TALKER',           'required': True},
        {'filename': 'subtalker_step.onnx', 'role': 'CODE_PREDICTOR',   'required': True},
        {'filename': 'codec_embed.onnx',    'role': 'SPEECH_TOKENIZER', 'required': True},
        {'filename': 'code2wav.onnx',       'role': 'VOCODER',          'required': True},
        {'filename': 'speaker_encoder.onnx','role': 'SPEAKER_ENCODER',  'required': False},
    ],
    'tokenizer': {'files': ['vocab.json', 'merges.txt', 'text_cond_table.f16',
        'subtalker_codec_embed.f16', 'subtalker_heads.f16',
        'baked_voices.bin', 'voices.json', 'ar_config.json']},
    'voices': [{'id': n, 'locale': 'ru-RU', 'displayName': n} for n in names],
}
json.dump(model_json, open(f'{OUT}/model.json','w'), ensure_ascii=False, indent=2)
print('wrote model.json with', len(names), 'voices')


## Ячейка 7: Сборка архива для устройства

In [ ]:
import zipfile
keep = ['talker_step.onnx','subtalker_step.onnx','codec_embed.onnx','code2wav.onnx','speaker_encoder.onnx',
        'text_cond_table.f16','text_cond_meta.json',
        'subtalker_codec_embed.f16','subtalker_heads.f16','subtalker_heads_bias.f16',
        'baked_voices.bin','voices.json','ar_config.json','model.json','vocab.json','merges.txt']
zip_path = '/content/qwen3-tts-ar-android.zip'
total = 0
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in keep:
        p = os.path.join(OUT, f)
        if os.path.exists(p):
            mb = os.path.getsize(p)/1024**2; total += mb
            print(f'  {f:30s} {mb:8.1f} MB'); zf.write(p, f)
        else:
            print(f'  {f:30s}  MISSING')
print(f'  total {total:.0f} MB -> {zip_path}')
try:
    from google.colab import files; files.download(zip_path)
except Exception: pass
